# DocViVQA Reasoning Pipeline

End-to-end pipeline for parsing questions, retrieving OCR cells, executing reasoning operations, and generating grounded predictions.


## 1. Foundations

Configure execution, load OCR data, and recover document structure.


### 1.1 Dependencies

Import the standard-library and third-party packages used throughout the pipeline.


In [1]:
import os

# Must be set before PyTorch initializes CUDA.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

from dataclasses import dataclass, field
from functools import lru_cache
from itertools import product
import json
from pathlib import Path
import re
import time
from typing import Iterable
import zipfile

import cv2
import numpy as np
import torch
from PIL import Image
from torch import nn
from torchvision.models import resnet18


### 1.2 Runtime & Data

Resolve paths, select the split, and configure deterministic execution.


In [2]:
# Phải đặt trước khi torch khởi tạo CUDA.

# Cố định kết quả giữa các lần chạy: cùng seed -> cùng file nộp.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

ROOT = Path.cwd().resolve().parent
if not (ROOT / "data").is_dir():
    raise FileNotFoundError(
        "Không tìm thấy data. Hãy chạy notebook từ thư mục notebooks của project DocViVQA."
    )

DATA = ROOT / 'data'
TRAIN_DIR = DATA / 'training_set'
RUNS = ROOT / 'outputs'
RUNS.mkdir(parents=True, exist_ok=True)
SPLIT = 'training_set'  # Đổi thành 'public_test' hoặc 'private_test' khi cần.

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'[cấu hình] ROOT={ROOT} | SPLIT={SPLIT}')
print('[cấu hình] thiết bị: ' + (f'{DEVICE} ({torch.cuda.get_device_name(0)})'
      if DEVICE.type == 'cuda' else str(DEVICE)))


[cấu hình] ROOT=E:\AIO\Project\olp-ai-ptit-2026-preliminary-round\DocViVQA | SPLIT=training_set
[cấu hình] thiết bị: cuda (NVIDIA GeForce RTX 3050 Laptop GPU)


### 1.3 Shared Primitives

Load document layouts and provide numeric and evidence utilities.


In [3]:
# OCR, document-layout, and shared formatting utilities.

@dataclass
class DocumentLayout:
    document_id: str
    blocks: list[dict]
    image_paths: list[Path] = field(default_factory=list)

def read_jsonl(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

def load_split(split_dir: Path) -> tuple[list[dict], dict[str, DocumentLayout]]:
    manifests = read_jsonl(split_dir / "manifest.jsonl")
    questions = read_jsonl(split_dir / "questions.jsonl")
    layouts: dict[str, DocumentLayout] = {}
    for manifest in manifests:
        payload = json.loads((split_dir / manifest["ocr_path"]).read_text(encoding="utf-8"))
        layouts[manifest["id"]] = DocumentLayout(
            document_id=manifest["id"],
            blocks=[block for page in payload["pages"] for block in page["blocks"]],
            image_paths=[split_dir / path for path in manifest["image_paths"]],
        )
    return questions, layouts

def center(block: dict) -> tuple[float, float]:
    x1, y1, x2, y2 = block["bbox"]
    return (x1 + x2) / 2, (y1 + y2) / 2

def block_width(block: dict) -> float:
    return block["bbox"][2] - block["bbox"][0]

def parse_number(text: str) -> float | None:
    value = text.strip().replace(" ", "").rstrip("%")
    if not re.fullmatch(r"[+-]?[0-9][0-9.,]*", value):
        return None
    if "," in value:
        value = value.replace(".", "").replace(",", ".")
    else:
        value = value.replace(".", "")
    try:
        return float(value)
    except ValueError:
        return None

def format_number(value: float) -> str:
    if abs(value - round(value)) <= 1e-9:
        return str(int(round(value)))
    return f"{value:.2f}".rstrip("0").rstrip(".").replace(".", ",")

def page_blocks(layout: DocumentLayout, page: int) -> list[dict]:
    return [block for block in layout.blocks if int(block["page"]) == page]


### 1.4 Table Geometry

Localize tables, recover rows, and map headers to cells.


In [4]:
def table_blocks(layout: DocumentLayout, page: int, table_index: int) -> list[dict]:
    blocks = page_blocks(layout, page)
    titles = sorted(
        [
            block for block in blocks
            if block_width(block) >= 0.65 and "BẢNG" in str(block["text"]).upper()
        ],
        key=lambda item: item["bbox"][1],
    )
    if not titles:
        return blocks if table_index == 1 else []
    if not 1 <= table_index <= len(titles):
        return []
    start = titles[table_index - 1]["bbox"][1] - 1e-6
    end = titles[table_index]["bbox"][1] - 1e-6 if table_index < len(titles) else 1.0
    return [block for block in blocks if start <= center(block)[1] < end]

def group_rows(blocks: list[dict]) -> list[list[dict]]:
    grouped: dict[float, list[dict]] = {}
    for block in blocks:
        grouped.setdefault(round(float(block["bbox"][1]), 6), []).append(block)
    return [sorted(values, key=lambda item: center(item)[0]) for _, values in sorted(grouped.items())]

def header_block(blocks: list[dict], text: str) -> dict | None:
    candidates = [block for block in blocks if str(block["text"]) == text]
    return min(candidates, key=lambda item: item["bbox"][1]) if candidates else None

def cell_under(row: list[dict], header: dict) -> dict | None:
    header_x, _ = center(header)
    candidates = [
        block for block in row
        if block["bbox"][0] - 1e-6 <= header_x <= block["bbox"][2] + 1e-6
    ]
    if candidates:
        return min(candidates, key=lambda item: abs(center(item)[0] - header_x))
    return min(row, key=lambda item: abs(center(item)[0] - header_x)) if row else None

def data_rows(blocks: list[dict], headers: list[dict]) -> list[list[dict]]:
    if not headers:
        return []
    boundary = max(header["bbox"][3] for header in headers)
    return [
        row for row in group_rows(blocks)
        if min(block["bbox"][1] for block in row) >= boundary - 1e-6
        and any(block["bbox"][3] - block["bbox"][1] < 0.06 for block in row)
    ]


## 2. Question Understanding

Convert Vietnamese questions into solver-ready structured intents.


### 2.1 Structured Intent

Define the normalized contract consumed by every solver.


In [5]:
@dataclass(frozen=True)
class StructuredIntent:
    reasoning_type: str
    template: str | None
    fields: dict
    route_source: str

REQUIRED_FIELDS = {
    "lookup": {"table", "page", "target_column", "condition_pairs"},
    "cross_page_sum": {"operands"},
    "count": {"table", "page", "condition_pairs"},
    "sum": {"table", "page", "target_column", "condition_pairs"},
    "argmax": {"table", "page", "return_column", "value_column"},
    "argmin": {"table", "page", "return_column", "value_column"},
    "compare": {"table", "page", "value_column", "row_condition_groups"},
    "visual_bold_lookup": {"table", "page", "target_column", "row_condition_groups"},
}


### 2.2 Language Patterns

Define ordered question patterns for every supported reasoning type.


In [6]:
# Ordered patterns for the supported reasoning types.
# Specific visual and cross-page patterns must precede general patterns.
PATTERN_CATALOG = {
    "visual_bold_lookup": {
        "visual_ocr_missing_format": re.compile(
            r"OCR không ghi nhận định dạng chữ\. Trong ảnh của bảng "
            r"(?P<table>\d+) ở trang (?P<page>\d+), hãy chọn dòng in đậm giữa "
            r"(?P<condition_text>.+?), rồi đọc ô thuộc cột (?P<target_column>.+?)\."
        ),
        "visual_presentation": re.compile(
            r"Chỉ dựa vào hình thức trình bày trên ảnh ở bảng "
            r"(?P<table>\d+) ở trang (?P<page>\d+): xác định dòng có toàn bộ chữ "
            r"in đậm trong hai dòng (?P<condition_text>.+?); giá trị "
            r"(?P<target_column>.+?) của dòng đó là gì\?"
        ),
        "visual_different_font": re.compile(
            r"Tại bảng (?P<table>\d+) ở trang (?P<page>\d+), hai dòng "
            r"(?P<condition_text>.+?) có kiểu chữ khác nhau\. Hãy tìm dòng in đậm "
            r"trên ảnh và trả về nội dung cột (?P<target_column>.+?)\."
        ),
        "visual_direct_observation": re.compile(
            r"Quan sát trực tiếp kiểu chữ trong ảnh tại bảng "
            r"(?P<table>\d+) ở trang (?P<page>\d+)\. Giữa dòng "
            r"(?P<condition_text>.+?), dòng nào được in đậm\? "
            r"Trả lời bằng giá trị ở cột (?P<target_column>.+?)\."
        ),
    },
    "cross_page_sum": {
        "cross_page_sum_standard": re.compile(
            r"Lấy (?P<target_column_1>.+?) của dòng có "
            r"(?P<row_1_condition_text>.+?) ở trang (?P<page_1>\d+) cộng với "
            r"(?P<target_column_2>.+?) của dòng có "
            r"(?P<row_2_condition_text>.+?) ở trang (?P<page_2>\d+)\. "
            r"Kết quả là bao nhiêu\?"
        ),
    },
    "compare": {
        "compare_two_rows": re.compile(
            r"So sánh tại bảng (?P<table>\d+) ở trang (?P<page>\d+): giữa dòng có "
            r"(?P<row_1_condition_text>.+?) với dòng có "
            r"(?P<row_2_condition_text>.+?), dòng nào có "
            r"(?P<value_column>.+?) cao hơn\?"
        ),
    },
    "sum": {
        "sum_two_rows": re.compile(
            r"Trong bảng (?P<table>\d+) ở trang (?P<page>\d+), tổng "
            r"(?P<target_column>.+?) của hai dòng có "
            r"(?P<condition_text>.+?) là bao nhiêu\?"
        ),
    },
    "count": {
        "count_rows": re.compile(
            r"Có bao nhiêu dòng trong bảng (?P<table>\d+) ở trang (?P<page>\d+) "
            r"có (?P<condition_column>.+?) là “(?P<condition_value>[^”]+)”\?"
        ),
    },
    "argmax": {
        "argmax_standard": re.compile(
            r"Trong bảng (?P<table>\d+) ở trang (?P<page>\d+), "
            r"(?P<return_column>.+?) nào có (?P<value_column>.+?) lớn nhất\?"
        ),
    },
    "argmin": {
        "argmin_standard": re.compile(
            r"Trong bảng (?P<table>\d+) ở trang (?P<page>\d+), "
            r"(?P<return_column>.+?) nào có (?P<value_column>.+?) nhỏ nhất\?"
        ),
    },
    "lookup": {
        "lookup_of_row": re.compile(
            r"Trong bảng (?P<table>\d+) ở trang (?P<page>\d+), "
            r"(?P<target_column>.+?) của dòng có (?P<condition_text>.+?) là gì\?"
        ),
        "lookup_tell_me": re.compile(
            r"Hãy cho biết (?P<target_column>.+?) tại bảng "
            r"(?P<table>\d+) ở trang (?P<page>\d+) đối với "
            r"(?P<condition_text>.+?)\."
        ),
        "lookup_row_records": re.compile(
            r"Tại bảng (?P<table>\d+) ở trang (?P<page>\d+), dòng "
            r"(?P<condition_text>.+?) ghi (?P<target_column>.+?) bằng bao nhiêu\?"
        ),
    },
}


### 2.3 Parsing Internals

Extract, normalize, and validate fields required by each reasoning type.


In [7]:
def parse_condition_pairs(text: str) -> list[tuple[str, str]]:
    matches = list(re.finditer(r"“([^”]+)”", text))
    pairs: list[tuple[str, str]] = []
    previous_end = 0
    for match in matches:
        header = text[previous_end : match.start()].strip()
        header = re.sub(r"^(?:và|với)\s+", "", header, flags=re.IGNORECASE)
        header = re.sub(r"^(?:dòng(?:\s+có)?|đối\s+với)\s+", "", header, flags=re.IGNORECASE)
        header = header.strip(" ,:.;")
        if not header:
            return []
        pairs.append((header, match.group(1)))
        previous_end = match.end()
    return pairs

def strong_template_match(question: str):
    question = question.strip()
    for reasoning_type, templates in PATTERN_CATALOG.items():
        for template_name, pattern in templates.items():
            match = pattern.fullmatch(question)
            if match:
                return reasoning_type, template_name, match.groupdict()
    return None

def extract_table(question: str) -> int | None:
    match = re.search(r"\bbảng\s+(\d+)", question, re.IGNORECASE)
    return int(match.group(1)) if match else None

# Single-page extractors only; cross_page_sum needs page_1/page_2 separately.
def extract_page(question: str) -> int | None:
    match = re.search(r"\btrang\s+(\d+)", question, re.IGNORECASE)
    return int(match.group(1)) if match else None

def extract_condition_pairs(text: str) -> list[tuple[str, str]]:
    pairs = parse_condition_pairs(text.strip().rstrip(" ?."))
    return [(re.sub(r"\s+là$", "", column, flags=re.IGNORECASE), value) for column, value in pairs]

def extract_lookup_target(question: str) -> str | None:
    patterns = (
        r"(?:hãy\s+)?cho\s+biết\s+(.+?)\s+(?:tại|trong)\s+bảng",
        r"(?:trong|tại)\s+bảng\s+\d+.*?,\s*(.+?)\s+của\s+dòng",
        r"\bghi\s+(.+?)\s+bằng\s+bao\s+nhiêu",
    )
    for pattern in patterns:
        match = re.search(pattern, question, re.IGNORECASE)
        if match:
            return match.group(1).strip(" ,:.;")
    return None

def extract_sum_target(question: str) -> str | None:
    match = re.search(r"\btổng\s+(.+?)\s+(?:của|trong)\s+", question, re.IGNORECASE)
    if not match:
        return None
    target = match.group(1).strip(" ,:.;")
    return None if re.match(r"^(?:trong|tại)\s+bảng\b", target, re.IGNORECASE) else target

def extract_condition_text(question: str, reasoning_type: str) -> str | None:
    patterns = {
        "lookup": (
            r"\bđối\s+với\s+(.+?)(?:[?.]|$)",
            r"\bdòng\s+có\s+(.+?)\s+là\s+gì(?:\?|$)",
            r"\bdòng\s+(.+?)\s+ghi\s+.+?\s+bằng\s+bao\s+nhiêu",
        ),
        "count": (r"\btrang\s+\d+\s+có\s+(.+?)(?:\?|\.|$)",),
        "sum": (r"\b(?:hai\s+)?dòng\s+có\s+(.+?)(?:\s+là\s+bao\s+nhiêu|\?|\.|$)",),
    }
    for pattern in patterns.get(reasoning_type, ()):
        match = re.search(pattern, question, re.IGNORECASE)
        if match:
            return match.group(1).strip(" ,:.;")
    return None

def extract_extreme_columns(question: str) -> tuple[str, str] | None:
    match = re.search(
        r"([^,?:]+?)\s+nào\s+có\s+(.+?)\s+(?:lớn|nhỏ)\s+nhất",
        question,
        re.IGNORECASE,
    )
    return (match.group(1).strip(), match.group(2).strip()) if match else None

def extract_compare_fields(question: str) -> dict:
    match = re.search(
        r"giữa\s+dòng\s+có\s+(.+?)\s+với\s+dòng\s+có\s+(.+?),\s*"
        r"dòng\s+nào\s+có\s+(.+?)\s+cao\s+hơn",
        question,
        re.IGNORECASE,
    )
    if not match:
        return {}
    return {
        "row_condition_groups": [
            extract_condition_pairs(match.group(1)),
            extract_condition_pairs(match.group(2)),
        ],
        "value_column": match.group(3).strip(),
    }

def parse_fields_by_type(question: str, reasoning_type: str) -> dict:
    """Relaxed recovery: extract required fields independently, never invent them."""
    if reasoning_type not in REQUIRED_FIELDS:
        return {}

    fields = {
        "table": extract_table(question),
        "page": extract_page(question),
    }
    condition_text = extract_condition_text(question, reasoning_type)
    if condition_text:
        fields["condition_pairs"] = extract_condition_pairs(condition_text)
    if reasoning_type == "lookup":
        fields["target_column"] = extract_lookup_target(question)
    elif reasoning_type == "sum":
        fields["target_column"] = extract_sum_target(question)
    elif reasoning_type in {"argmax", "argmin"}:
        columns = extract_extreme_columns(question)
        if columns:
            fields["return_column"], fields["value_column"] = columns
    elif reasoning_type == "compare":
        fields.update(extract_compare_fields(question))
    return {key: value for key, value in fields.items() if value not in (None, [], "")}

def normalize_fields(reasoning_type: str, raw_fields: dict) -> dict:
    """Map strong-template fields and relaxed fields to one solver-facing schema."""
    if reasoning_type not in REQUIRED_FIELDS:
        return dict(raw_fields)

    if reasoning_type == "visual_bold_lookup":
        pairs = extract_condition_pairs(raw_fields.get("condition_text", ""))
        target = raw_fields.get("target_column")
        table, page = raw_fields.get("table"), raw_fields.get("page")
        split_at = next((index for index, pair in enumerate(pairs[1:], 1) if pair[0] == pairs[0][0]), None) if pairs else None
        if not target or not table or not page or split_at is None:
            return {}
        return {
            "table": int(table), "page": int(page), "target_column": target.strip(),
            "row_condition_groups": [pairs[:split_at], pairs[split_at:]],
        }

    if reasoning_type == "cross_page_sum":
        operands = []
        for index in (1, 2):
            target = raw_fields.get(f"target_column_{index}")
            page = raw_fields.get(f"page_{index}")
            conditions = extract_condition_pairs(raw_fields.get(f"row_{index}_condition_text", ""))
            if not target or not page or not conditions:
                return {}
            operands.append({
                "page": int(page), "table": 1,
                "target_column": target.strip(), "condition_pairs": conditions,
            })
        return {"operands": operands}

    normalized = {}
    for name in ("table", "page"):
        value = raw_fields.get(name)
        if value not in (None, ""):
            normalized[name] = int(value)

    for name in ("target_column", "return_column", "value_column"):
        value = raw_fields.get(name)
        if value:
            normalized[name] = value.strip()

    pairs = raw_fields.get("condition_pairs")
    if pairs is None and raw_fields.get("condition_text"):
        pairs = extract_condition_pairs(raw_fields["condition_text"])
    if pairs is None and raw_fields.get("condition_column") and raw_fields.get("condition_value"):
        pairs = [(raw_fields["condition_column"], raw_fields["condition_value"])]
    if pairs:
        normalized["condition_pairs"] = list(pairs)

    groups = raw_fields.get("row_condition_groups")
    if groups is None and raw_fields.get("row_1_condition_text") and raw_fields.get("row_2_condition_text"):
        groups = [
            extract_condition_pairs(raw_fields["row_1_condition_text"]),
            extract_condition_pairs(raw_fields["row_2_condition_text"]),
        ]
    if groups and len(groups) == 2 and all(groups):
        normalized["row_condition_groups"] = [list(group) for group in groups]
    return normalized

def intent_is_complete(intent: StructuredIntent) -> bool:
    required = REQUIRED_FIELDS.get(intent.reasoning_type)
    return required is None or required <= intent.fields.keys()

def heuristic_route(question: str) -> str | None:
    """Only route when the operation signal is explicit."""
    q = question.strip().lower()
    if "in đậm" in q or "kiểu chữ" in q:
        return "visual_bold_lookup"
    if q.startswith("lấy ") and "cộng với" in q:
        return "cross_page_sum"
    if q.startswith("so sánh tại bảng"):
        return "compare"
    if q.startswith(("có bao nhiêu dòng", "đếm số dòng")):
        return "count"
    if "tổng " in q and "dòng" in q:
        return "sum"
    if q.endswith("lớn nhất?"):
        return "argmax"
    if q.endswith("nhỏ nhất?"):
        return "argmin"
    # Generic lookup rule must remain after cross-page, compare, and sum checks.
    if q.startswith(("hãy cho biết", "cho biết", "tại bảng")) or " của dòng có " in q:
        return "lookup"
    return None


### 2.4 Intent Routing

Prefer exact templates, then use deterministic heuristics.


In [8]:
def parse_intent(question: str) -> StructuredIntent:
    matched = strong_template_match(question)
    if matched:
        reasoning_type, template, raw_fields = matched
        fields = normalize_fields(reasoning_type, raw_fields)
        return StructuredIntent(reasoning_type, template, fields, "strong_template")

    reasoning_type = heuristic_route(question)
    if reasoning_type is None:
        return StructuredIntent("unmatched", None, {}, "none")

    raw_fields = parse_fields_by_type(question, reasoning_type)
    fields = normalize_fields(reasoning_type, raw_fields)
    return StructuredIntent(reasoning_type, None, fields, "heuristic")


## 3. Retrieval & Deterministic Reasoning

Resolve relevant cells and execute symbolic operations.


### 3.1 Candidate Retrieval

Match condition pairs to rows and resolve the target cell.


In [9]:
def matching_rows(blocks: list[dict], pairs: list[tuple[str, str]]) -> list[tuple[list[dict], list[dict]]]:
    if not pairs:
        return []
    headers: list[dict] = []
    for header_text, _ in pairs:
        header = header_block(blocks, header_text)
        if header is None:
            return []
        headers.append(header)
    matches: list[tuple[list[dict], list[dict]]] = []
    for row in data_rows(blocks, headers):
        cells = [cell_under(row, header) for header in headers]
        if all(cell is not None and str(cell["text"]) == value for cell, (_, value) in zip(cells, pairs, strict=True)):
            matches.append((row, [cell for cell in cells if cell is not None]))
    return matches

def intent_location(fields: dict) -> tuple[int, int]:
    """Read the common table/page fields produced by PATTERN_CATALOG."""
    return int(fields["table"]), int(fields["page"])

def resolve_target_cell(
    layout: DocumentLayout,
    *,
    page: int,
    table: int,
    target_column: str,
    condition_pairs: list[tuple[str, str]],
) -> tuple[dict, list[dict]] | None:
    blocks = table_blocks(layout, page, table)
    matches = matching_rows(blocks, condition_pairs)
    target_header = header_block(blocks, target_column)
    if len(matches) != 1 or target_header is None:
        return None

    row, condition_cells = matches[0]
    answer_cell = cell_under(row, target_header)
    return (answer_cell, condition_cells + [answer_cell]) if answer_cell else None


### 3.2 Evidence Grounding

Build submission evidence directly from the OCR cells used to derive each answer.


In [10]:
def build_evidence(blocks: Iterable[dict]) -> list[dict]:
    seen: set[str] = set()
    result: list[dict] = []
    for block in blocks:
        if block["block_id"] in seen:
            continue
        seen.add(block["block_id"])
        result.append({"page": block["page"], "bbox": block["bbox"]})
    return result


### 3.3 Lookup

Read the cell at the resolved row-column intersection.


In [11]:
def solve_lookup(fields: dict, layout: DocumentLayout) -> tuple[str, list[dict]] | None:
    required = {"table", "page", "target_column", "condition_pairs"}
    if not required <= fields.keys():
        return None

    resolved = resolve_target_cell(layout, **fields)
    if resolved is None:
        return None
    answer_cell, evidence = resolved
    return str(answer_cell["text"]), build_evidence(evidence)


### 3.4 Aggregation

Count matching rows or sum values from resolved cells.


In [12]:
def solve_count(fields: dict, layout: DocumentLayout) -> tuple[str, list[dict]] | None:
    required = {"table", "page", "condition_pairs"}
    if not required <= fields.keys():
        return None

    table_index, page = intent_location(fields)
    blocks = table_blocks(layout, page, table_index)
    matches = matching_rows(blocks, fields["condition_pairs"])
    evidence = [cell for _, cells in matches for cell in cells]
    return str(len(matches)), build_evidence(evidence)

def split_two_conditions(blocks: list[dict], pairs: list[tuple[str, str]]):
    for split_at in range(1, len(pairs)):
        first = matching_rows(blocks, pairs[:split_at])
        second = matching_rows(blocks, pairs[split_at:])
        if len(first) == len(second) == 1 and first[0][0] != second[0][0]:
            return first[0], second[0]
    return None

def solve_sum(fields: dict, layout: DocumentLayout) -> tuple[str, list[dict]] | None:
    required = {"table", "page", "target_column", "condition_pairs"}
    if not required <= fields.keys():
        return None

    table_index, page = intent_location(fields)
    blocks = table_blocks(layout, page, table_index)
    rows = split_two_conditions(blocks, fields["condition_pairs"])
    target_header = header_block(blocks, fields["target_column"])
    if rows is None or target_header is None:
        return None

    first_cell = cell_under(rows[0][0], target_header)
    second_cell = cell_under(rows[1][0], target_header)
    first = parse_number(first_cell["text"]) if first_cell else None
    second = parse_number(second_cell["text"]) if second_cell else None
    if first is None or second is None:
        return None

    evidence = rows[0][1] + rows[1][1] + [first_cell, second_cell]
    return format_number(first + second), build_evidence(evidence)


### 3.5 Ranking & Comparison

Reconstruct logical rows and select values across candidates.


In [13]:
def reconstruct_logical_rows(
    blocks: list[dict],
    headers: list[dict],
) -> list[tuple[list[dict | None], list[dict]]]:
    """Map merged OCR cells to every physical row their bbox actually covers."""
    physical_rows = data_rows(blocks, headers)
    header_bottom = max(header["bbox"][3] for header in headers)
    logical_rows = []

    for physical_row in physical_rows:
        row_top = min(block["bbox"][1] for block in physical_row)
        row_bottom = min(
            block["bbox"][3]
            for block in physical_row
            if block["bbox"][3] > row_top
        )
        row_y = (row_top + row_bottom) / 2
        row = sorted([
            block for block in blocks
            if block not in headers
            and block["bbox"][1] >= header_bottom - 1e-6
            and block["bbox"][1] - 1e-6 <= row_y <= block["bbox"][3] + 1e-6
        ], key=lambda block: center(block)[0])
        cells = []

        for header in headers:
            header_x, _ = center(header)
            candidates = [
                block for block in row
                if block["bbox"][0] - 1e-6 <= header_x <= block["bbox"][2] + 1e-6
            ]
            cells.append(
                min(candidates, key=lambda block: abs(center(block)[0] - header_x))
                if candidates else None
            )

        logical_rows.append((cells, row))

    return logical_rows

def solve_argextreme(fields: dict, layout: DocumentLayout, select) -> tuple[str, list[dict]] | None:
    table_index, page = intent_location(fields)
    blocks = table_blocks(layout, page, table_index)
    return_header = header_block(blocks, fields["return_column"])
    value_header = header_block(blocks, fields["value_column"])
    if return_header is None or value_header is None:
        return None

    candidates = []
    for (return_cell, value_cell), row in reconstruct_logical_rows(
        blocks,
        [return_header, value_header],
    ):
        value = parse_number(value_cell["text"]) if value_cell else None
        if return_cell is not None and value is not None:
            candidates.append((value, return_cell, value_cell, row))
    if not candidates:
        return None

    _, return_cell, value_cell, winner_row = select(candidates, key=lambda item: item[0])
    value_x, _ = center(value_header)
    prefixes = [
        [block for block in row if center(block)[0] < value_x - 1e-6]
        for _, _, _, row in candidates
    ]
    winner_prefix = [block for block in winner_row if center(block)[0] < value_x - 1e-6]
    evidence = winner_prefix
    for size in range(1, len(winner_prefix) + 1):
        signature = tuple(str(block["text"]) for block in winner_prefix[:size])
        matching_ids = {
            tuple(block["block_id"] for block in prefix[:size])
            for prefix in prefixes
            if tuple(str(block["text"]) for block in prefix[:size]) == signature
        }
        if len(matching_ids) == 1:
            evidence = winner_prefix[:size]
            break
    return str(return_cell["text"]), build_evidence(evidence + [value_cell])

def solve_argmax(fields: dict, layout: DocumentLayout):
    return solve_argextreme(fields, layout, max)

def solve_argmin(fields: dict, layout: DocumentLayout):
    return solve_argextreme(fields, layout, min)

def solve_compare(fields: dict, layout: DocumentLayout) -> tuple[str, list[dict]] | None:
    table_index, page = intent_location(fields)
    blocks = table_blocks(layout, page, table_index)
    value_header = header_block(blocks, fields["value_column"])
    if value_header is None:
        return None

    matched = [matching_rows(blocks, group) for group in fields["row_condition_groups"]]
    if any(len(rows) != 1 for rows in matched):
        return None

    first, second = matched[0][0], matched[1][0]
    first_value_cell = cell_under(first[0], value_header)
    second_value_cell = cell_under(second[0], value_header)
    first_value = parse_number(first_value_cell["text"]) if first_value_cell else None
    second_value = parse_number(second_value_cell["text"]) if second_value_cell else None
    if first_value is None or second_value is None or first_value == second_value:
        return None

    winner = first if first_value > second_value else second
    answer_cell = winner[1][0]
    evidence = first[1] + second[1] + [first_value_cell, second_value_cell]
    return str(answer_cell["text"]), build_evidence(evidence)


### 3.6 Cross-page Aggregation

Reuse target-cell resolution across multiple pages.


In [14]:
def solve_cross_page_sum(fields: dict, layout: DocumentLayout) -> tuple[str, list[dict]] | None:
    operands = fields.get("operands", [])
    if len(operands) != 2:
        return None

    resolved = [resolve_target_cell(layout, **operand) for operand in operands]
    if any(item is None for item in resolved):
        return None

    cells = [item[0] for item in resolved]
    values = [parse_number(str(cell["text"])) for cell in cells]
    if any(value is None for value in values):
        return None

    evidence = [block for _, blocks in resolved for block in blocks]
    return format_number(sum(values)), build_evidence(evidence)


## 4. Visual Reasoning

Resolve bold-row questions with progressively stronger visual signals.


### 4.1 Stroke-width Heuristics

Measure ink width and vote across rows and cells.


In [15]:
VISUAL_SCORE_MARGIN = 0.02
VISUAL_VOTE_MARGIN = 4

@lru_cache(maxsize=64)
def grayscale_page(path: Path) -> np.ndarray:
    return np.asarray(Image.open(path).convert("L"))

def cell_stroke_width(cell: dict, image: np.ndarray) -> float | None:
    height, width = image.shape
    x1, y1, x2, y2 = cell["bbox"]
    left, right = int(x1 * width), int(x2 * width)
    top, bottom = int(y1 * height), int(y2 * height)
    crop = image[top:bottom, left:right]
    if crop.size == 0 or min(crop.shape) < 5:
        return None

    _, ink = cv2.threshold(crop, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    border = max(1, round(min(ink.shape) * 0.03))
    ink[:border, :] = ink[-border:, :] = 0
    ink[:, :border] = ink[:, -border:] = 0
    if int((ink > 0).sum()) < 10:
        return None

    distance = cv2.distanceTransform(ink, cv2.DIST_L2, 3)
    return float(2 * distance[ink > 0].mean())

def row_boldness_score(row: list[dict], image: np.ndarray) -> float | None:
    scores = [cell_stroke_width(cell, image) for cell in row]
    scores = [score for score in scores if score is not None]
    return float(np.median(scores)) if scores else None

def choose_bolder_row(scores: list[float | None], margin: float = VISUAL_SCORE_MARGIN) -> int | None:
    if len(scores) != 2 or any(score is None for score in scores):
        return None
    if abs(scores[0] - scores[1]) < margin:
        return None
    return int(scores[1] > scores[0])

def choose_bolder_vote(first: list[float | None], second: list[float | None]) -> int | None:
    votes = [0, 0]
    for first_score, second_score in zip(first, second):
        if first_score is None or second_score is None:
            continue
        votes[0] += first_score > second_score
        votes[1] += second_score > first_score
    if abs(votes[0] - votes[1]) < VISUAL_VOTE_MARGIN:
        return None
    return int(votes[1] > votes[0])


### 4.2 Pairwise ResNet18

Load the trained fallback for visually ambiguous candidates.


In [16]:
BOLD_PAIR_CHECKPOINT = ROOT / "artifacts" / "models" / "bold_pair_resnet18.pt"
PAIR_CONFIDENCE_THRESHOLD = 0.60

class BoldPairResNet18(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = resnet18(weights=None)
        self.encoder = nn.Sequential(*list(backbone.children())[:-1])
        self.head = nn.Linear(1024, 2)

    def forward(self, first: torch.Tensor, second: torch.Tensor) -> torch.Tensor:
        first_features = self.encoder(first).flatten(1)
        second_features = self.encoder(second).flatten(1)
        return self.head(torch.cat([first_features, second_features], dim=1))

@lru_cache(maxsize=1)
def load_bold_pair_model() -> tuple[BoldPairResNet18, int, dict] | None:
    if not BOLD_PAIR_CHECKPOINT.exists():
        return None
    checkpoint = torch.load(BOLD_PAIR_CHECKPOINT, map_location=DEVICE, weights_only=True)
    if checkpoint.get("architecture") != "pair_resnet18_frozen":
        return None
    crop_preprocessing = checkpoint.get("crop_preprocessing", {})
    if crop_preprocessing.get("kind") != "cell_inset_composite":
        return None
    model = BoldPairResNet18().to(DEVICE)
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()
    return model, int(checkpoint["image_size"]), crop_preprocessing

def row_pair_tensor(image_path: Path, row: list[dict], image_size: int, crop_preprocessing: dict) -> torch.Tensor:
    image = Image.open(image_path).convert("L")
    width, height = image.size
    row_left = int(min(block["bbox"][0] for block in row) * width)
    row_top = int(min(block["bbox"][1] for block in row) * height)
    row_right = int(max(block["bbox"][2] for block in row) * width)
    row_bottom = int(max(block["bbox"][3] for block in row) * height)
    crop = Image.new("L", (row_right - row_left, row_bottom - row_top), color=255)
    inset_x = float(crop_preprocessing["x_inset"])
    inset_y = float(crop_preprocessing["y_inset"])
    for block in row:
        x1, y1, x2, y2 = block["bbox"]
        left = int((x1 + (x2 - x1) * inset_x) * width)
        top = int((y1 + (y2 - y1) * inset_y) * height)
        right = int((x2 - (x2 - x1) * inset_x) * width)
        bottom = int((y2 - (y2 - y1) * inset_y) * height)
        if right > left and bottom > top:
            crop.paste(image.crop((left, top, right, bottom)), (left - row_left, top - row_top))
    crop = crop.resize((image_size, image_size), Image.Resampling.BILINEAR)
    tensor = torch.from_numpy(np.asarray(crop, dtype=np.float32) / 255.0).unsqueeze(0).repeat(3, 1, 1)
    return (tensor - 0.5) / 0.5

def pairwise_bold_winner(layout: DocumentLayout, page: int, rows: list[list[dict]]) -> int | None:
    loaded = load_bold_pair_model()
    if loaded is None or len(rows) != 2 or not 1 <= page <= len(layout.image_paths):
        return None
    model, image_size, crop_preprocessing = loaded
    image_path = layout.image_paths[page - 1]
    with torch.inference_mode():
        logits = model(
            row_pair_tensor(image_path, rows[0], image_size, crop_preprocessing).unsqueeze(0).to(DEVICE),
            row_pair_tensor(image_path, rows[1], image_size, crop_preprocessing).unsqueeze(0).to(DEVICE),
        )
        probabilities = logits.softmax(dim=1)[0]
    winner = int(probabilities.argmax().item())
    return winner if float(probabilities[winner]) >= PAIR_CONFIDENCE_THRESHOLD else None


### 4.3 Progressive Resolution

Apply row statistics, cell voting, then ResNet18.


In [17]:
def solve_visual_bold_lookup(fields: dict, layout: DocumentLayout) -> tuple[str, list[dict]] | None:
    required = {"table", "page", "target_column", "row_condition_groups"}
    if not required <= fields.keys() or len(fields["row_condition_groups"]) != 2:
        return None

    table_index, page = intent_location(fields)
    blocks = table_blocks(layout, page, table_index)
    target_header = header_block(blocks, fields["target_column"])
    matched = [matching_rows(blocks, group) for group in fields["row_condition_groups"]]
    if target_header is None or any(len(rows) != 1 for rows in matched):
        return None
    if not 1 <= page <= len(layout.image_paths):
        return None

    candidates = [rows[0] for rows in matched]
    image = grayscale_page(layout.image_paths[page - 1])
    cell_scores = [[cell_stroke_width(cell, image) for cell in row] for row, _ in candidates]
    winner = choose_bolder_row([row_boldness_score(row, image) for row, _ in candidates])
    if winner is None:
        winner = choose_bolder_vote(*cell_scores)
    if winner is None:
        winner = pairwise_bold_winner(layout, page, [row for row, _ in candidates])
    if winner is None:
        return None

    row, _ = candidates[winner]
    answer_cell = cell_under(row, target_header)
    evidence = [cell for _, condition_cells in candidates for cell in condition_cells]
    return (str(answer_cell["text"]), build_evidence(evidence + [answer_cell])) if answer_cell else None


## 5. Orchestration & Output

Dispatch solvers, verify behavior, and emit submission artifacts.


### 5.1 Solver Dispatcher

Map reasoning types to deterministic solver functions.


In [18]:
SOLVERS = {
    "lookup": solve_lookup,
    "cross_page_sum": solve_cross_page_sum,
    "visual_bold_lookup": solve_visual_bold_lookup,
    "count": solve_count,
    "sum": solve_sum,
    "argmax": solve_argmax,
    "argmin": solve_argmin,
    "compare": solve_compare,
}

def solve(intent, layout: DocumentLayout):
    solver = SOLVERS.get(intent.reasoning_type)
    if solver is None or not intent_is_complete(intent):
        return None
    return solver(intent.fields, layout)


### 5.2 Regression Checks

Validate representative parsing and solver behavior.


In [19]:
STRONG_CASES = {
    "lookup": "Hãy cho biết Hiện có tại bảng 1 ở trang 1 đối với Phòng ban “Công nghệ”.",
    "count": "Có bao nhiêu dòng trong bảng 1 ở trang 1 có Cây trồng là “Lúa”?",
    "sum": "Trong bảng 1 ở trang 1, tổng Tuyển mới của hai dòng có Phòng ban “Kinh doanh” và Phòng ban “Kế toán” là bao nhiêu?",
}

RELAXED_CASES = {
    "lookup": "Cho biết Hiện có trong bảng 1 trang 1 đối với Phòng ban “Công nghệ”.",
    "count": "Đếm số dòng ở bảng 1 trang 1 có Cây trồng là “Lúa”.",
    "sum": "Hãy tính tổng Tuyển mới trong bảng 1 trang 1 của hai dòng có Phòng ban “Kinh doanh” và Phòng ban “Kế toán”.",
}

INCOMPLETE_CASES = {
    "lookup": "Cho biết Hiện có đối với Phòng ban “Công nghệ”.",
    "count": "Đếm số dòng ở bảng 1 có Cây trồng là “Lúa”.",
    "sum": "Hãy tính tổng trong bảng 1 trang 1 của hai dòng có Phòng ban “Kinh doanh” và Phòng ban “Kế toán”.",
}

for expected, question in STRONG_CASES.items():
    intent = parse_intent(question)
    assert intent.reasoning_type == expected
    assert intent.route_source == "strong_template"
    assert intent_is_complete(intent)
    assert isinstance(intent.fields["table"], int)
    assert "condition_pairs" in intent.fields

for expected, question in RELAXED_CASES.items():
    intent = parse_intent(question)
    assert intent.reasoning_type == expected
    assert intent.route_source == "heuristic"
    assert intent_is_complete(intent), intent

for expected, question in INCOMPLETE_CASES.items():
    intent = parse_intent(question)
    assert intent.reasoning_type == expected
    assert not intent_is_complete(intent), intent

print("3 strong + 3 relaxed + 3 incomplete intents: OK")

NEW_SOLVER_CASES = {
    "argmax": "Trong bảng 1 ở trang 1, Phòng ban nào có Tuyển mới lớn nhất?",
    "argmin": "Trong bảng 1 ở trang 1, Phòng ban nào có Hiện có nhỏ nhất?",
    "compare": "So sánh tại bảng 1 ở trang 1: giữa dòng có Phòng ban “Kinh doanh” với dòng có Phòng ban “Kế toán”, dòng nào có Tuyển mới cao hơn?",
}

for expected, question in NEW_SOLVER_CASES.items():
    intent = parse_intent(question)
    assert intent.reasoning_type == expected
    assert intent_is_complete(intent), intent

print("argmax + argmin + compare intents: OK")


def cross_page_sum_regression_test() -> None:
    headers = [
        ("Khoa", [0.0, 0.0, 0.5, 0.1]),
        ("Lượt khám", [0.5, 0.0, 1.0, 0.1]),
    ]
    blocks = []
    for page, value in ((1, "666"), (2, "633")):
        for index, (text, bbox) in enumerate(headers):
            blocks.append({"block_id": f"p{page}_h{index}", "page": page, "text": text, "bbox": bbox})
        blocks.extend([
            {"block_id": f"p{page}_k", "page": page, "text": "Cần Thơ", "bbox": [0.0, 0.1, 0.5, 0.12]},
            {"block_id": f"p{page}_v", "page": page, "text": value, "bbox": [0.5, 0.1, 1.0, 0.12]},
        ])
    result = solve_cross_page_sum({"operands": [
        {"page": 1, "table": 1, "target_column": "Lượt khám", "condition_pairs": [("Khoa", "Cần Thơ")]},
        {"page": 2, "table": 1, "target_column": "Lượt khám", "condition_pairs": [("Khoa", "Cần Thơ")]},
    ]}, DocumentLayout("cross-page-test", blocks))
    assert result and result[0] == "1299"

def visual_bold_regression_test():
    assert choose_bolder_row([2.28, 2.09]) == 0
    assert choose_bolder_row([2.10, 2.11]) is None
    assert choose_bolder_vote([2.20, 2.18, 2.11, 2.22, 2.17], [2.14, 2.13, 2.16, 2.15, 2.14]) is None
    assert choose_bolder_vote([2.20, 2.18, 2.19, 2.22, 2.17], [2.14, 2.13, 2.16, 2.15, 2.14]) == 0


def argextreme_evidence_regression_test():
    blocks = [
        {"block_id": "h0", "page": 1, "text": "Đơn vị", "bbox": [0.0, 0.0, 0.4, 0.1]},
        {"block_id": "h2", "page": 1, "text": "Giá trị", "bbox": [0.7, 0.0, 1.0, 0.1]},
        {"block_id": "a0", "page": 1, "text": "A", "bbox": [0.0, 0.1, 0.4, 0.2]},
        {"block_id": "a1", "page": 1, "text": "10", "bbox": [0.4, 0.1, 0.7, 0.2]},
        {"block_id": "a2", "page": 1, "text": "2", "bbox": [0.7, 0.1, 1.0, 0.2]},
        {"block_id": "b0", "page": 1, "text": "A", "bbox": [0.0, 0.2, 0.4, 0.3]},
        {"block_id": "b1", "page": 1, "text": "20", "bbox": [0.4, 0.2, 0.7, 0.3]},
        {"block_id": "b2", "page": 1, "text": "5", "bbox": [0.7, 0.2, 1.0, 0.3]},
    ]
    result = solve_argmax({"table": 1, "page": 1, "return_column": "Đơn vị", "value_column": "Giá trị"}, DocumentLayout("argmax-test", blocks))
    assert result and result[0] == "A" and len(result[1]) == 3


cross_page_sum_regression_test()
visual_bold_regression_test()
argextreme_evidence_regression_test()


3 strong + 3 relaxed + 3 incomplete intents: OK
argmax + argmin + compare intents: OK


AssertionError: 

### 5.3 End-to-End Inference

Parse each question, solve it, and write grounded predictions.


In [ ]:
PREDICTIONS_PATH = RUNS / f"predictions_{SPLIT}.jsonl"
questions, layouts = load_split(DATA / SPLIT)

solved = 0
with PREDICTIONS_PATH.open("w", encoding="utf-8") as handle:
    for item in questions:
        intent = parse_intent(item["question"])
        result = solve(intent, layouts[item["document_id"]])

        answer, evidence = result if result else ("không xác định", [])
        solved += int(result is not None)
        handle.write(json.dumps({
            "question_id": item["question_id"],
            "answer": answer,
            "evidence": evidence,
        }, ensure_ascii=False, sort_keys=True) + "\n")

print(f"Trả lời được {solved}/{len(questions)} câu; file: {PREDICTIONS_PATH}")


Trả lời được 11000/11000 câu; file: E:\AIO\Project\olp-ai-ptit-2026-preliminary-round\DocViVQA\outputs\predictions_training_set.jsonl


### 5.4 Submission Packaging

Archive predictions in the required submission layout.


In [ ]:
SUBMISSION_PATH = RUNS / f'submission_{SPLIT}.zip'

with zipfile.ZipFile(SUBMISSION_PATH, 'w', zipfile.ZIP_DEFLATED) as archive:
    archive.write(PREDICTIONS_PATH, 'predictions.jsonl')
print(f'[nộp bài] đã tạo {SUBMISSION_PATH}')


[nộp bài] đã tạo E:\AIO\Project\olp-ai-ptit-2026-preliminary-round\DocViVQA\outputs\submission_training_set.zip
